# 06 — Fault isolation and recovery

Branch-A populations are independent of branch B. Trap and recovery counts remain separate. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

rows=[]
for experiment, artifact, env in [('e-iso-4','containment.json','E_ISO_4_DIR'),('e-iso-7','branch-isolation.json','E_ISO_7_DIR')]:
    try: batch=resolve_result_batch(experiment, diagnostic_path=os.environ.get(env))
    except (FileNotFoundError, RuntimeError, ValueError): continue
    for path, value in passed_json(batch, artifact):
        if experiment == 'e-iso-4':
            recoveries=sum(int(node.get('recovery_count',0)) for node in value.get('nodes',[]))
            rows.append({'question':'epoch recovery','condition':value.get('condition'),'traps':value.get('traps_total',0),'recoveries':recoveries,'branch_a_throughput_msg_s':None,'branch_a_p95_ns':None})
        else:
            branch=value['branches']['branch_a']
            rows.append({'question':'branch-A throughput and latency','condition':value.get('condition'),'traps':None,'recoveries':None,'branch_a_throughput_msg_s':branch['throughput']['mean_messages_per_second'],'branch_a_p95_ns':branch['latency_ns']['p95']})
if rows:
    df=pd.DataFrame(rows); print(evidence_label(len(df), 'events, messages/second, nanoseconds', False)); display(df)
else:
    display(pd.DataFrame([pending_record('branch-A throughput and epoch recovery','no passed focused isolation leaf','mixed; labelled per column')]))
